# Additional Analyses: C, E, F, G

Standalone notebook extending `payman_post_analyses_CLEAN.ipynb`.

- **C** — Clinical scale (SHAPS / TEPS / DASS) correlations with significant brain hierarchy metrics
- **E** — Raw COVtau-derived hierarchy as a simpler alternative to GEC-based metrics
- **F** — Group-level GEC matrix visualization and comparison
- **G** — SC→GEC divergence per subject: does high anhedonia show more idiosyncratic effective connectivity?

In [ ]:
import numpy as np
import scipy.io as sio
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import ranksums, pearsonr
from scipy.linalg import solve, LinAlgError
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

color_low  = '#1f77b4'
color_high = '#ff7f0e'

In [ ]:
def compute_hierarchy(Ceff):
    """
    Compute trophic coherence (tc) and hierarchical levels (hl) from a
    directed weighted connectivity matrix. Port of MATLAB compute_hierarchy.
    Returns (np.nan, None) on singular system.
    """
    A = Ceff.T
    d     = A.sum(axis=0)
    delta = A.sum(axis=1)
    u = d + delta
    v = d - delta

    Lambda = np.diag(u) - A - A.T
    Lambda[0, 0] = 0.0  # regularisation for singularity

    try:
        gamma = solve(Lambda, v)
    except (LinAlgError, np.linalg.LinAlgError):
        return np.nan, np.full(Ceff.shape[0], np.nan)

    gamma -= gamma.min()
    H  = (gamma[:, None] - gamma[None, :] - 1) ** 2
    denom = A.sum()
    if denom == 0:
        return np.nan, gamma
    F0 = np.sum(A * H) / denom
    return 1.0 - F0, gamma


def styled_boxplot(ax, data_low, data_high, labels=('Low', 'High'),
                   ylabel='', title='', p_ols=None):
    """Standard two-group boxplot matching existing notebook style."""
    bp = ax.boxplot([data_low, data_high],
                    tick_labels=list(labels), patch_artist=True)
    bp['boxes'][0].set_facecolor(color_low)
    bp['boxes'][1].set_facecolor(color_high)
    for m in bp['medians']:
        m.set_color('black')
        m.set_linewidth(1.5)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    if p_ols is not None:
        ylo, yhi = ax.get_ylim()
        ax.text(1.5, yhi - (yhi - ylo) * 0.05,
                f'OLS p = {p_ols:.4f}', ha='center', va='top', fontsize=10)

In [ ]:
# ── PATHS ──────────────────────────────────────────────────────────────────
BASE      = '/Users/proghani/Documents/personal/local_thesis_proj/TCP_thesis_project/my_analysis/trophic_coherence_analysis'
ANHED_DIR = '/Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/anhedonia/NEW_4_factor_clustering_ipnybV4'
OUT_DIR   = f'{BASE}/outputs/NEW_4_factor_clustering_ipnybV4'
LABEL_FILE = '/Users/proghani/Documents/personal/code_experiments/neuro/phd_stuff/tcp_parcellations/data/atlases/Schaefer2018_200Parcels_7Networks_order_Tian_Subcortex_S2_label.txt'
DEMOS_PATH = f'{BASE}/data/demos_payman.xls'
PHENO_DIR  = '/Users/proghani/Documents/personal/local_thesis_proj/TCP_thesis_project/phenotype/imputed/sum_subscale'
SC_CSV     = '/Users/proghani/Documents/personal/local_thesis_proj/TCP_thesis_project/my_analysis/NEMO/SC_files_Jakub/SC_schaefer200_tian_S2_7Networks_32fold_groupconnectome_3T_MNI152NLin2009cAsym_2mm.csv'

# ── SUBJECT IDs ─────────────────────────────────────────────────────────────
mat_low  = sio.loadmat(f'{ANHED_DIR}/low_anhedonia.mat')
mat_high = sio.loadmat(f'{ANHED_DIR}/high_anhedonia.mat')
ids_low  = [mat_low['low_anhedonia'][i, 1].flat[0]  for i in range(mat_low['low_anhedonia'].shape[0])]
ids_high = [mat_high['high_anhedonia'][i, 1].flat[0] for i in range(mat_high['high_anhedonia'].shape[0])]
print(f'N LOW={len(ids_low)}, N HIGH={len(ids_high)}')

# ── PER-GROUP .mat RESULTS ──────────────────────────────────────────────────
data_low  = sio.loadmat(f'{OUT_DIR}/low_anhedonia/results_Ceff_low_anhedonia.mat')
data_high = sio.loadmat(f'{OUT_DIR}/high_anhedonia/results_Ceff_high_anhedonia.mat')

Ceff_LOW              = data_low['Ceff_LOW']               # (139, 232, 232)
Ceff_HIGH             = data_high['Ceff_HIGH']             # (71,  232, 232)
COVtau_LOW            = data_low['COVtau_LOW']             # (139, 232, 232)
COVtau_HIGH           = data_high['COVtau_HIGH']           # (71,  232, 232)
Ceffgroup_LOW         = data_low['Ceffgroup_LOW']          # (232, 232)
Ceffgroup_HIGH        = data_high['Ceffgroup_HIGH']        # (232, 232)
hierarchicallevels_LOW  = data_low['hierarchicallevels_LOW']    # (139, 232)
hierarchicallevels_HIGH = data_high['hierarchicallevels_HIGH']  # (71,  232)
trophiccoherence_LOW  = data_low['trophiccoherence_LOW'].flatten()
trophiccoherence_HIGH = data_high['trophiccoherence_HIGH'].flatten()

NSUB_LOW  = Ceff_LOW.shape[0]
NSUB_HIGH = Ceff_HIGH.shape[0]

# ── DEMOGRAPHICS ────────────────────────────────────────────────────────────
demos = pd.read_excel(DEMOS_PATH, header=1)
demos['Age_clean']  = demos['Age'].replace(999, np.nan)
demos['is_patient'] = demos['Primary_Dx_Payman'].apply(lambda x: 0 if x == 999 else 1)
demos['anhedonia_group'] = None
demos.loc[demos['subjectkey'].isin(ids_low),  'anhedonia_group'] = 'LOW'
demos.loc[demos['subjectkey'].isin(ids_high), 'anhedonia_group'] = 'HIGH'

# ── REGION LABELS ───────────────────────────────────────────────────────────
with open(LABEL_FILE, 'r') as f:
    lines = f.read().strip().splitlines()
region_labels = [lines[i] for i in range(0, len(lines), 2)]
assert len(region_labels) == 232

# ── NETWORK INDICES ─────────────────────────────────────────────────────────
network_indices = {}
for i, label in enumerate(region_labels):
    net = 'Subcortex' if i < 32 else label.split('_')[2]
    network_indices.setdefault(net, []).append(i)

display_order = ['Vis', 'SomMot', 'DorsAttn', 'SalVentAttn', 'Limbic', 'Cont', 'Default', 'Subcortex']

subcortical_idx = np.arange(0, 32)
cortical_idx    = np.arange(32, 232)

# ── REWARD ROI INDICES ──────────────────────────────────────────────────────
reward_rois = {
    'NAc':     [8, 9, 24, 25],
    'Caudate': [14, 15, 30, 31],
    'Putamen': [12, 13, 28, 29],
}
reward_rois['OFC']        = [i for i, l in enumerate(region_labels) if 'Limbic_OFC' in l or 'Cont_OFC' in l]
reward_rois['mPFC/vmPFC'] = [i for i, l in enumerate(region_labels) if 'Default_PFC' in l and 'pCun' not in l]
reward_rois['ACC']        = [i for i, l in enumerate(region_labels) if 'SalVentAttn_Med' in l or 'Cont_Cing' in l]
reward_rois['dlPFC']      = [i for i, l in enumerate(region_labels) if 'Cont_PFCl' in l]

pfc_idx      = reward_rois['dlPFC'] + reward_rois['mPFC/vmPFC'] + reward_rois['OFC'] + reward_rois['ACC']
striatum_idx = reward_rois['NAc'] + reward_rois['Caudate'] + reward_rois['Putamen']

print('All data loaded.')

---
## Analysis C: Clinical Scale Correlations with Brain Hierarchy Metrics

Test whether continuous symptom scores (SHAPS, TEPS-AP, TEPS-CP, DASS-Depression, DASS-Anxiety, DASS-Stress) predict the brain hierarchy metrics that showed significant anhedonia-group differences in the primary analyses: top-down flow dominance, frontostriatal asymmetry, and trophic levels in prefrontal reward regions (OFC, mPFC/vmPFC, ACC, dlPFC).

**Strategy:** OLS regression — `brain_metric ~ clinical_scale_z + Age_clean + sex + is_patient` across all 210 subjects. Clinical scales are z-scored before entry so beta coefficients are directly comparable across scales with different ranges. FDR correction is applied across all 36 tests (6 scales × 6 metrics) as a single family.

In [ ]:
# %% C1 — Load clinical scale summary scores
# %% C1 — Load clinical scale summary scores
shaps = (
    pd.read_csv(f'{PHENO_DIR}/tot_imputed_shaps01.csv', header=0)[
        ['subjectkey', 'shaps_total']
    ]
)

teps = (
    pd.read_csv(f'{PHENO_DIR}/tot_imputed_teps01.csv', header=0)[
        ['subjectkey',
         'teps_total']
    ]
)

dass = (
    pd.read_csv(f'{PHENO_DIR}/tot_imputed_dass01.csv', header=0)[
        ['subjectkey', 'dass_p1_3', 'dass_p1_10', 'dass_p2_10', 'dass_p2_3', 'dass_p1_16']
    ]
    .assign(dass_anhedonia=lambda x: x.iloc[:, 1:].sum(axis=1))
    [['subjectkey', 'dass_anhedonia']]
)

df_clinical = shaps.merge(teps, on='subjectkey').merge(dass, on='subjectkey')

print(f'Clinical data loaded: {len(df_clinical)} subjects')
print(df_clinical[
    ['shaps_total', 'teps_total', 'dass_anhedonia']
].describe().round(2))



In [ ]:
# %% C2 — Recompute brain metrics for all 210 subjects combined
# Metrics: td_dominance, fs_asym, and mean trophic level in 4 prefrontal ROIs

td_all = np.zeros(NSUB_LOW + NSUB_HIGH)
fs_all = np.zeros(NSUB_LOW + NSUB_HIGH)

for s in range(NSUB_LOW):
    C = Ceff_LOW[s]
    td_all[s] = (np.sum(C[np.ix_(subcortical_idx, cortical_idx)]) -
                 np.sum(C[np.ix_(cortical_idx, subcortical_idx)]))
    fs_all[s] = (np.sum(C[np.ix_(striatum_idx, pfc_idx)]) -
                 np.sum(C[np.ix_(pfc_idx, striatum_idx)]))

for s in range(NSUB_HIGH):
    C = Ceff_HIGH[s]
    td_all[NSUB_LOW + s] = (np.sum(C[np.ix_(subcortical_idx, cortical_idx)]) -
                             np.sum(C[np.ix_(cortical_idx, subcortical_idx)]))
    fs_all[NSUB_LOW + s] = (np.sum(C[np.ix_(striatum_idx, pfc_idx)]) -
                             np.sum(C[np.ix_(pfc_idx, striatum_idx)]))

# Prefrontal ROI mean trophic levels
hl_all = np.vstack([hierarchicallevels_LOW, hierarchicallevels_HIGH])  # (210, 232)
pfc_rois = ['OFC', 'mPFC/vmPFC', 'ACC', 'dlPFC']

df_brain = pd.DataFrame({
    'subjectkey':     ids_low + ids_high,
    'anhedonia_group': ['LOW'] * NSUB_LOW + ['HIGH'] * NSUB_HIGH,
    'td_dominance':   td_all,
    'fs_asym':        fs_all,
    'ofc_hl':         np.mean(hl_all[:, reward_rois['OFC']],        axis=1),
    'mpfc_hl':        np.mean(hl_all[:, reward_rois['mPFC/vmPFC']], axis=1),
    'acc_hl':         np.mean(hl_all[:, reward_rois['ACC']],        axis=1),
    'dlpfc_hl':       np.mean(hl_all[:, reward_rois['dlPFC']],      axis=1),
})

print(f'Brain metrics: {df_brain.shape}')
print(df_brain[['td_dominance','fs_asym','ofc_hl','mpfc_hl','acc_hl','dlpfc_hl']].describe().round(4))

In [ ]:
# %% C3 — Build master OLS dataframe; z-score clinical scales

df_master_c = (df_brain
               .merge(df_clinical, on='subjectkey', how='inner')
               .merge(demos[['subjectkey', 'Age_clean', 'sex', 'is_patient']],
                      on='subjectkey', how='left'))

df_master_c['sex'] = df_master_c['sex'].replace('O', np.nan)
df_master_c['anhedonia_group'] = pd.Categorical(df_master_c['anhedonia_group'],
                                                 categories=['LOW', 'HIGH'])
df_master_c['sex'] = pd.Categorical(df_master_c['sex'], categories=['F', 'M'])

# Z-score clinical scales so betas are comparable across scales
scale_cols = ['shaps_total', 'teps_total', 'dass_anhedonia']
for col in scale_cols:
    df_master_c[col] = (df_master_c[col] - df_master_c[col].mean()) / df_master_c[col].std()

print(f'Master dataframe: {len(df_master_c)} rows ({df_master_c["anhedonia_group"].value_counts().to_dict()})')
print(f'sex=O excluded in OLS listwise deletion (4 subjects)')

In [ ]:
# %% C4 — OLS loop: 6 metrics × 6 scales = 36 tests, single FDR family

metric_cols = ['td_dominance', 'fs_asym', 'ofc_hl', 'mpfc_hl', 'acc_hl', 'dlpfc_hl']
metric_labels = ['TD-Dominance', 'FS-Asymmetry', 'OFC-HL', 'mPFC-HL', 'ACC-HL', 'dlPFC-HL']
scale_labels  = ['shaps_total', 'teps_total', 'dass_anhedonia']

records = []
for metric in metric_cols:
    for scale in scale_cols:
        formula = f'{metric} ~ {scale} + Age_clean + sex + is_patient'
        res = smf.ols(formula, data=df_master_c).fit()
        records.append({
            'metric': metric,
            'scale':  scale,
            'beta':   res.params[scale],
            'ci_lo':  res.conf_int().loc[scale, 0],
            'ci_hi':  res.conf_int().loc[scale, 1],
            'p_raw':  res.pvalues[scale],
            'N':      int(res.nobs),
        })

df_results_c = pd.DataFrame(records)
_, p_fdr, _, _ = multipletests(df_results_c['p_raw'], method='fdr_bh')
df_results_c['p_fdr'] = p_fdr
df_results_c['sig']   = p_fdr < 0.05

print('=== Analysis C: OLS Results (sorted by p_fdr) ===')
print(df_results_c[['metric','scale','beta','ci_lo','ci_hi','p_raw','p_fdr','sig']]
      .sort_values('p_fdr').to_string(index=False, float_format='%.4f'))
print(f'\n{df_results_c["sig"].sum()} of 36 tests FDR-significant (q<0.05)')

In [ ]:
# %% C5 — Sensitivity: patients-only OLS for any FDR-significant findings

sig_pairs = df_results_c[df_results_c['sig']][['metric', 'scale']].values.tolist()

if len(sig_pairs) == 0:
    print('No FDR-significant findings in full sample — patients-only sensitivity skipped.')
else:
    df_patients = df_master_c[df_master_c['is_patient'] == 1].copy()
    print(f'Patients-only N = {len(df_patients)} ({df_patients["anhedonia_group"].value_counts().to_dict()})')
    print()
    for metric, scale in sig_pairs:
        formula = f'{metric} ~ {scale} + Age_clean + sex'  # drop is_patient
        res_p = smf.ols(formula, data=df_patients).fit()
        beta_p = res_p.params[scale]
        p_p    = res_p.pvalues[scale]
        survives = '✓ survives' if p_p < 0.05 else '✗ attenuated'
        print(f'  {metric:<15} ~ {scale:<15}: β={beta_p:+.4f}, p={p_p:.4f}  [{survives}]')

In [ ]:
# %% C6 — Figure: heatmap of OLS beta coefficients

n_metrics = len(metric_cols)
n_scales  = len(scale_cols)

# Build beta and significance matrices
beta_mat = np.zeros((n_metrics, n_scales))
sig_mat  = np.zeros((n_metrics, n_scales), dtype=bool)

for i, metric in enumerate(metric_cols):
    for j, scale in enumerate(scale_cols):
        row = df_results_c[(df_results_c['metric'] == metric) &
                            (df_results_c['scale']  == scale)].iloc[0]
        beta_mat[i, j] = row['beta']
        sig_mat[i, j]  = row['sig']

vmax = np.nanmax(np.abs(beta_mat)) * 1.05  # symmetric colormap

fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(beta_mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
plt.colorbar(im, ax=ax, label='OLS β  (z-scored clinical scale)')

# Star annotations for FDR-significant cells
for i in range(n_metrics):
    for j in range(n_scales):
        txt = f'{beta_mat[i,j]:+.3f}'
        if sig_mat[i, j]:
            txt += ' *'
        ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                color='black' if abs(beta_mat[i,j]) < vmax * 0.6 else 'white')

ax.set_xticks(range(n_scales))
ax.set_xticklabels(scale_labels, rotation=45, ha='right', fontsize=11)
ax.set_yticks(range(n_metrics))
ax.set_yticklabels(metric_labels, fontsize=11)
ax.set_title('Clinical Scale × Brain Hierarchy Metric: OLS β Coefficients\n'
             '(* = FDR-corrected p < 0.05; controlling for Age, Sex, Patient/HC)',
             fontsize=12)
fig.tight_layout()
plt.show()


In [ ]:
# %% C_DX — OLS loop with Dx_broad instead of is_patient (HC / Mood / Anxiety_Trauma / Other)

# --- Build Dx_broad and attach to a copy of df_master_c ---
dx_map = {
    'MDD': 'Mood', 'BP1': 'Mood', 'BP2': 'Mood',
    'Dysthymia': 'Mood', 'Other Mood Dis': 'Mood',
    'GAD': 'Anxiety_Trauma', 'PTSD': 'Anxiety_Trauma',
    'SAD': 'Anxiety_Trauma', 'Other Anxiety': 'Anxiety_Trauma',
    'SZ': 'Other', 'SZA': 'Other',
    'OCD': 'Other', 'SUD': 'Other', 'ADHD': 'Other', 'ED': 'Other',
}
demos_dx = demos[['subjectkey', 'Primary_Dx_Payman']].copy()
demos_dx['Dx_broad'] = (demos_dx['Primary_Dx_Payman']
                        .replace(999, 'HC')
                        .astype(str).str.strip()
                        .map(dx_map)
                        .fillna('HC'))  # HC won't match dx_map → fillna

df_master_dx = df_master_c.merge(demos_dx[['subjectkey', 'Dx_broad']], on='subjectkey', how='left')
df_master_dx['Dx_broad'] = pd.Categorical(
    df_master_dx['Dx_broad'],
    categories=['HC', 'Mood', 'Anxiety_Trauma', 'Other']
)
print('Dx_broad distribution:')
print(df_master_dx['Dx_broad'].value_counts())

# --- OLS loop: same 36 tests, C(Dx_broad) replaces is_patient ---
n_metrics = len(metric_cols)
n_scales  = len(scale_cols)

records_dx = []
for metric in metric_cols:
    for scale in scale_cols:
        formula = f'{metric} ~ {scale} + Age_clean + sex + Dx_broad'
        res = smf.ols(formula, data=df_master_dx.dropna(subset=['Dx_broad'])).fit()
        records_dx.append({
            'metric': metric,
            'scale':  scale,
            'beta':   res.params[scale],
            'ci_lo':  res.conf_int().loc[scale, 0],
            'ci_hi':  res.conf_int().loc[scale, 1],
            'p_raw':  res.pvalues[scale],
            'N':      int(res.nobs),
        })

df_results_dx = pd.DataFrame(records_dx)
_, p_fdr_dx, _, _ = multipletests(df_results_dx['p_raw'], method='fdr_bh')
df_results_dx['p_fdr'] = p_fdr_dx
df_results_dx['sig']   = p_fdr_dx < 0.05

print('\n=== Analysis C (Dx_broad): OLS Results (sorted by p_fdr) ===')
print(df_results_dx[['metric','scale','beta','ci_lo','ci_hi','p_raw','p_fdr','sig']]
      .sort_values('p_fdr').to_string(index=False, float_format='%.4f'))
print(f'\n{df_results_dx["sig"].sum()} of {len(df_results_dx)} tests FDR-significant (q<0.05)')

# --- Heatmap ---
beta_mat_dx = np.zeros((n_metrics, n_scales))
sig_mat_dx  = np.zeros((n_metrics, n_scales), dtype=bool)

for i, metric in enumerate(metric_cols):
    for j, scale in enumerate(scale_cols):
        row = df_results_dx[(df_results_dx['metric'] == metric) &
                             (df_results_dx['scale']  == scale)].iloc[0]
        beta_mat_dx[i, j] = row['beta']
        sig_mat_dx[i, j]  = row['sig']

vmax_dx = np.nanmax(np.abs(beta_mat_dx)) * 1.05

fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(beta_mat_dx, cmap='RdBu_r', vmin=-vmax_dx, vmax=vmax_dx, aspect='auto')
plt.colorbar(im, ax=ax, label='OLS β  (z-scored clinical scale)')

for i in range(n_metrics):
    for j in range(n_scales):
        txt = f'{beta_mat_dx[i,j]:+.3f}'
        if sig_mat_dx[i, j]:
            txt += ' *'
        ax.text(j, i, txt, ha='center', va='center', fontsize=9,
                color='black' if abs(beta_mat_dx[i,j]) < vmax_dx * 0.6 else 'white')

ax.set_xticks(range(n_scales))
ax.set_xticklabels(scale_labels, rotation=45, ha='right', fontsize=11)
ax.set_yticks(range(n_metrics))
ax.set_yticklabels(metric_labels, fontsize=11)
ax.set_title('Clinical Scale × Brain Hierarchy Metric: OLS β Coefficients\n'
             '(* = FDR-corrected p < 0.05; controlling for Age, Sex, Dx category [HC ref])',
             fontsize=12)
fig.tight_layout()
plt.show()


---
## Analysis E: Raw COVtau-Derived Hierarchy as a Simpler Alternative

The GEC is estimated by optimizing the Hopf model to fit both empirical FC and COVtau. Before this optimization, we already have the empirical COVtau matrices per subject. Here we test two simpler directedness measures derived directly from COVtau — without any model fitting — and compare their ability to distinguish the two anhedonia groups against the full GEC-based results.

**Metric 1: COVtau asymmetry score** — Frobenius norm of (COVtau − COVtauᵀ) per subject, normalized by N². Captures the overall temporal asymmetry of the raw covariance structure.

**Metric 2: COVtau-based trophic coherence** — Apply `compute_hierarchy()` to the positive part of each subject's COVtau matrix (negative entries clipped to zero). Directly comparable to the GEC-based trophic coherence.

In [ ]:
# %% E2 — COVtau asymmetry score (fully vectorized, no Python loop)
# ~27% of COVtau entries are negative; the antisymmetric part captures raw directionality

asym_LOW  = COVtau_LOW  - COVtau_LOW.transpose(0, 2, 1)   # (139, 232, 232)
frob_LOW  = np.sqrt(np.sum(asym_LOW ** 2, axis=(1, 2)))    # (139,)
cov_asym_LOW  = frob_LOW / (232 ** 2)

asym_HIGH = COVtau_HIGH - COVtau_HIGH.transpose(0, 2, 1)  # (71, 232, 232)
frob_HIGH = np.sqrt(np.sum(asym_HIGH ** 2, axis=(1, 2)))  # (71,)
cov_asym_HIGH = frob_HIGH / (232 ** 2)

_, p_asym_rs = ranksums(cov_asym_LOW, cov_asym_HIGH)
print('COVtau asymmetry score:')
print(f'  LOW  mean={cov_asym_LOW.mean():.6f}  std={cov_asym_LOW.std():.6f}')
print(f'  HIGH mean={cov_asym_HIGH.mean():.6f}  std={cov_asym_HIGH.std():.6f}')
print(f'  Ranksum p = {p_asym_rs:.4f}')

In [ ]:
# %% E3 — OLS for COVtau asymmetry score

df_ols_e_asym = pd.concat([
    pd.DataFrame({'subjectkey': ids_low,  'cov_asym': cov_asym_LOW,  'anhedonia_group': 'LOW'}),
    pd.DataFrame({'subjectkey': ids_high, 'cov_asym': cov_asym_HIGH, 'anhedonia_group': 'HIGH'}),
], ignore_index=True)
df_ols_e_asym = df_ols_e_asym.merge(
    demos[['subjectkey', 'Age_clean', 'sex', 'is_patient']], on='subjectkey', how='left')
df_ols_e_asym['sex'] = df_ols_e_asym['sex'].replace('O', np.nan)
df_ols_e_asym['anhedonia_group'] = pd.Categorical(df_ols_e_asym['anhedonia_group'], categories=['LOW', 'HIGH'])
df_ols_e_asym['sex']             = pd.Categorical(df_ols_e_asym['sex'],             categories=['F', 'M'])

res_e_asym = smf.ols('cov_asym ~ anhedonia_group + Age_clean + sex + is_patient',
                      data=df_ols_e_asym).fit()
coef_e_asym = res_e_asym.params['anhedonia_group[T.HIGH]']
p_e_asym    = res_e_asym.pvalues['anhedonia_group[T.HIGH]']
ci_e_asym   = res_e_asym.conf_int().loc['anhedonia_group[T.HIGH]']

print('OLS — COVtau asymmetry ~ Anhedonia Group + covariates')
print(f'  β = {coef_e_asym:+.6f}  [{ci_e_asym[0]:+.6f}, {ci_e_asym[1]:+.6f}]')
print(f'  p = {p_e_asym:.4f}  N = {int(res_e_asym.nobs)}')

# Sensitivity: patients only
df_pat_asym = df_ols_e_asym[df_ols_e_asym['is_patient'] == 1].copy()
res_e_asym_pat = smf.ols('cov_asym ~ anhedonia_group + Age_clean + sex', data=df_pat_asym).fit()
p_e_asym_pat = res_e_asym_pat.pvalues['anhedonia_group[T.HIGH]']
print(f'  Patients-only: β = {res_e_asym_pat.params["anhedonia_group[T.HIGH]"]:+.6f}, p = {p_e_asym_pat:.4f}')

In [ ]:
# %% E3_DX — Sensitivity: COVtau asymmetry ~ Anhedonia Group + Dx_broad (HC ref)

# Build Dx_broad if not already in scope (reuse if C_DX cell was run)
if 'demos_dx' not in dir():
    dx_map = {
        'MDD': 'Mood', 'BP1': 'Mood', 'BP2': 'Mood',
        'Dysthymia': 'Mood', 'Other Mood Dis': 'Mood',
        'GAD': 'Anxiety_Trauma', 'PTSD': 'Anxiety_Trauma',
        'SAD': 'Anxiety_Trauma', 'Other Anxiety': 'Anxiety_Trauma',
        'SZ': 'Other', 'SZA': 'Other',
        'OCD': 'Other', 'SUD': 'Other', 'ADHD': 'Other', 'ED': 'Other',
    }
    demos_dx = demos[['subjectkey', 'Primary_Dx_Payman']].copy()
    demos_dx['Dx_broad'] = (demos_dx['Primary_Dx_Payman']
                            .replace(999, 'HC')
                            .astype(str).str.strip()
                            .map(dx_map)
                            .fillna('HC'))

df_ols_e_asym_dx = df_ols_e_asym.merge(
    demos_dx[['subjectkey', 'Dx_broad']], on='subjectkey', how='left')
df_ols_e_asym_dx['Dx_broad'] = pd.Categorical(
    df_ols_e_asym_dx['Dx_broad'],
    categories=['HC', 'Mood', 'Anxiety_Trauma', 'Other']
)

res_e_asym_dx = smf.ols(
    'cov_asym ~ anhedonia_group + Age_clean + sex + Dx_broad',
    data=df_ols_e_asym_dx.dropna(subset=['Dx_broad'])
).fit()
coef_e_asym_dx = res_e_asym_dx.params['anhedonia_group[T.HIGH]']
p_e_asym_dx    = res_e_asym_dx.pvalues['anhedonia_group[T.HIGH]']
ci_e_asym_dx   = res_e_asym_dx.conf_int().loc['anhedonia_group[T.HIGH]']

print('OLS — COVtau asymmetry ~ Anhedonia Group + Dx_broad (HC ref)')
print(f'  β = {coef_e_asym_dx:+.6f}  [{ci_e_asym_dx[0]:+.6f}, {ci_e_asym_dx[1]:+.6f}]')
print(f'  p = {p_e_asym_dx:.4f}  N = {int(res_e_asym_dx.nobs)}')


In [ ]:
# %% E4 — COVtau-based trophic coherence
# Clip negative COVtau entries to zero before passing to compute_hierarchy.
# Rationale: trophic levels require non-negative edge weights. Positive COVtau
# entries reflect genuine lagged covariance in the forward time direction;
# negative entries (~27% of all entries) are discarded by this clip.

print('Computing COVtau-based trophic coherence (per-subject loop)...')

tc_covtau_LOW = np.zeros(NSUB_LOW)
for s in range(NSUB_LOW):
    C_clip = np.maximum(COVtau_LOW[s], 0.0)
    tc_covtau_LOW[s], _ = compute_hierarchy(C_clip)

tc_covtau_HIGH = np.zeros(NSUB_HIGH)
for s in range(NSUB_HIGH):
    C_clip = np.maximum(COVtau_HIGH[s], 0.0)
    tc_covtau_HIGH[s], _ = compute_hierarchy(C_clip)

n_nan = np.isnan(tc_covtau_LOW).sum() + np.isnan(tc_covtau_HIGH).sum()
print(f'  NaN subjects (singular system): {n_nan}')

_, p_covtc_rs = ranksums(tc_covtau_LOW[~np.isnan(tc_covtau_LOW)],
                         tc_covtau_HIGH[~np.isnan(tc_covtau_HIGH)])
print(f'COVtau-TC  LOW  mean={np.nanmean(tc_covtau_LOW):.4f}')
print(f'COVtau-TC  HIGH mean={np.nanmean(tc_covtau_HIGH):.4f}')
print(f'Ranksum p = {p_covtc_rs:.4f}')

In [ ]:
# %% E5 — OLS for COVtau-based trophic coherence

df_ols_e_tc = pd.concat([
    pd.DataFrame({'subjectkey': ids_low,  'tc_covtau': tc_covtau_LOW,  'anhedonia_group': 'LOW'}),
    pd.DataFrame({'subjectkey': ids_high, 'tc_covtau': tc_covtau_HIGH, 'anhedonia_group': 'HIGH'}),
], ignore_index=True)
df_ols_e_tc = df_ols_e_tc.merge(
    demos[['subjectkey', 'Age_clean', 'sex', 'is_patient']], on='subjectkey', how='left')
df_ols_e_tc['sex'] = df_ols_e_tc['sex'].replace('O', np.nan)
df_ols_e_tc['anhedonia_group'] = pd.Categorical(df_ols_e_tc['anhedonia_group'], categories=['LOW', 'HIGH'])
df_ols_e_tc['sex']             = pd.Categorical(df_ols_e_tc['sex'],             categories=['F', 'M'])

res_e_tc = smf.ols('tc_covtau ~ anhedonia_group + Age_clean + sex + is_patient',
                    data=df_ols_e_tc).fit()
coef_e_tc = res_e_tc.params['anhedonia_group[T.HIGH]']
p_e_tc    = res_e_tc.pvalues['anhedonia_group[T.HIGH]']
ci_e_tc   = res_e_tc.conf_int().loc['anhedonia_group[T.HIGH]']

print('OLS — COVtau TC ~ Anhedonia Group + covariates')
print(f'  β = {coef_e_tc:+.4f}  [{ci_e_tc[0]:+.4f}, {ci_e_tc[1]:+.4f}]')
print(f'  p = {p_e_tc:.4f}  N = {int(res_e_tc.nobs)}')

In [ ]:
# %% E5_DX — Sensitivity: COVtau TC ~ Anhedonia Group + Dx_broad (HC ref)

df_ols_e_tc_dx = df_ols_e_tc.merge(
    demos_dx[['subjectkey', 'Dx_broad']], on='subjectkey', how='left')
df_ols_e_tc_dx['Dx_broad'] = pd.Categorical(
    df_ols_e_tc_dx['Dx_broad'],
    categories=['HC', 'Mood', 'Anxiety_Trauma', 'Other']
)

res_e_tc_dx = smf.ols(
    'tc_covtau ~ anhedonia_group + Age_clean + sex + Dx_broad',
    data=df_ols_e_tc_dx.dropna(subset=['Dx_broad'])
).fit()
coef_e_tc_dx = res_e_tc_dx.params['anhedonia_group[T.HIGH]']
p_e_tc_dx    = res_e_tc_dx.pvalues['anhedonia_group[T.HIGH]']
ci_e_tc_dx   = res_e_tc_dx.conf_int().loc['anhedonia_group[T.HIGH]']

print('OLS — COVtau TC ~ Anhedonia Group + Dx_broad (HC ref)')
print(f'  β = {coef_e_tc_dx:+.4f}  [{ci_e_tc_dx[0]:+.4f}, {ci_e_tc_dx[1]:+.4f}]')
print(f'  p = {p_e_tc_dx:.4f}  N = {int(res_e_tc_dx.nobs)}')


In [ ]:
# %% E6 — Also run OLS for GEC metrics (for figure comparison)
# GEC trophic coherence (existing result)

df_ols_gec_tc = pd.concat([
    pd.DataFrame({'subjectkey': ids_low,  'gec_tc': trophiccoherence_LOW,  'anhedonia_group': 'LOW'}),
    pd.DataFrame({'subjectkey': ids_high, 'gec_tc': trophiccoherence_HIGH, 'anhedonia_group': 'HIGH'}),
], ignore_index=True)
df_ols_gec_tc = df_ols_gec_tc.merge(
    demos[['subjectkey', 'Age_clean', 'sex', 'is_patient']], on='subjectkey', how='left')
df_ols_gec_tc['sex'] = df_ols_gec_tc['sex'].replace('O', np.nan)
df_ols_gec_tc['anhedonia_group'] = pd.Categorical(df_ols_gec_tc['anhedonia_group'], categories=['LOW', 'HIGH'])
df_ols_gec_tc['sex']             = pd.Categorical(df_ols_gec_tc['sex'], categories=['F', 'M'])
res_gec_tc = smf.ols('gec_tc ~ anhedonia_group + Age_clean + sex + is_patient', data=df_ols_gec_tc).fit()
p_gec_tc   = res_gec_tc.pvalues['anhedonia_group[T.HIGH]']

# GEC top-down flow dominance (significant finding from primary analysis)
td_low  = td_all[:NSUB_LOW]
td_high = td_all[NSUB_LOW:]

df_ols_gec_td = pd.concat([
    pd.DataFrame({'subjectkey': ids_low,  'td_dom': td_low,  'anhedonia_group': 'LOW'}),
    pd.DataFrame({'subjectkey': ids_high, 'td_dom': td_high, 'anhedonia_group': 'HIGH'}),
], ignore_index=True)
df_ols_gec_td = df_ols_gec_td.merge(
    demos[['subjectkey', 'Age_clean', 'sex', 'is_patient']], on='subjectkey', how='left')
df_ols_gec_td['sex'] = df_ols_gec_td['sex'].replace('O', np.nan)
df_ols_gec_td['anhedonia_group'] = pd.Categorical(df_ols_gec_td['anhedonia_group'], categories=['LOW', 'HIGH'])
df_ols_gec_td['sex']             = pd.Categorical(df_ols_gec_td['sex'], categories=['F', 'M'])
res_gec_td = smf.ols('td_dom ~ anhedonia_group + Age_clean + sex + is_patient', data=df_ols_gec_td).fit()
p_gec_td   = res_gec_td.pvalues['anhedonia_group[T.HIGH]']

print(f'GEC trophic coherence  OLS p = {p_gec_tc:.4f}')
print(f'GEC top-down dominance OLS p = {p_gec_td:.4f}')

In [ ]:
# %% E7 — Figure: 2×2 comparison of GEC vs COVtau metrics

fig, axes = plt.subplots(2, 2, figsize=(10, 8))

# (0,0) GEC trophic coherence
styled_boxplot(axes[0, 0], trophiccoherence_LOW, trophiccoherence_HIGH,
               ylabel='Trophic coherence', title='GEC Trophic Coherence', p_ols=p_gec_tc)

# (0,1) COVtau trophic coherence
styled_boxplot(axes[0, 1], tc_covtau_LOW, tc_covtau_HIGH,
               ylabel='Trophic coherence', title='COVtau Trophic Coherence (raw)', p_ols=p_e_tc)

# (1,0) GEC top-down dominance (significant)
styled_boxplot(axes[1, 0], td_low, td_high,
               ylabel='Net top-down flow', title='GEC Top-Down Dominance', p_ols=p_gec_td)

# (1,1) COVtau asymmetry score
styled_boxplot(axes[1, 1], cov_asym_LOW, cov_asym_HIGH,
               ylabel='Frobenius asymmetry / N²',
               title='COVtau Asymmetry Score (raw)', p_ols=p_e_asym)

# Column headers
for ax, col_title in zip(axes[0], ['GEC-optimised', 'Raw COVtau (no model fitting)']):
    ax.set_title(f'{col_title}\n{ax.get_title()}', fontsize=10)

fig.suptitle('GEC-derived vs Raw COVtau Hierarchy Metrics\n'
             '(OLS p-values control for Age, Sex, Patient/HC)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

---
## Analysis F: Group-Level GEC Matrix Comparison

Visualise and compare the group-averaged effective connectivity matrices (`Ceffgroup_LOW` and `Ceffgroup_HIGH`), which represent the population-level directed connectivity scaffold for each group.

Panels:
1. Three heatmaps (LOW / HIGH / difference) sorted by Yeo-7 network + Subcortex
2. Edge-by-edge scatter (LOW vs HIGH), coloured by underlying SC strength
3. Group-level trophic levels per network, derived from each group Ceffgroup

In [ ]:
# %% F1 — Build network-sorted index and sorted matrices

# Sort regions by network membership (display_order)
sorted_idx = []
for net in display_order:
    sorted_idx.extend(network_indices[net])
sorted_idx = np.array(sorted_idx)

# Cumulative boundary positions
boundaries = []
cum = 0
for net in display_order:
    cum += len(network_indices[net])
    boundaries.append(cum)
mid_positions = [boundaries[i-1] if i > 0 else 0 for i in range(len(display_order))]
mid_positions = [(boundaries[i] + (boundaries[i-1] if i > 0 else 0)) / 2
                 for i in range(len(display_order))]

# Sorted GEC matrices
Ceff_low_s  = Ceffgroup_LOW[np.ix_(sorted_idx, sorted_idx)]
Ceff_high_s = Ceffgroup_HIGH[np.ix_(sorted_idx, sorted_idx)]
Ceff_diff_s = Ceff_high_s - Ceff_low_s

print(f'Sorted matrices: {Ceff_low_s.shape}')
print(f'Boundaries: {boundaries}')

In [ ]:
# %% F2 — Three heatmaps: LOW / HIGH / Difference

def draw_network_borders(ax, boundaries, color='white', lw=0.8):
    for b in boundaries[:-1]:  # skip the last (232 = edge of matrix)
        ax.axhline(b - 0.5, color=color, lw=lw)
        ax.axvline(b - 0.5, color=color, lw=lw)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- LOW ---
im0 = axes[0].imshow(Ceff_low_s, cmap='hot', vmin=0, vmax=0.2, aspect='auto')
draw_network_borders(axes[0], boundaries)
plt.colorbar(im0, ax=axes[0], label='GEC weight')
axes[0].set_title('LOW Anhedonia — Group GEC', fontsize=12)
axes[0].set_xticks(mid_positions)
axes[0].set_xticklabels(display_order, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(mid_positions)
axes[0].set_yticklabels(display_order, fontsize=8)

# --- HIGH ---
im1 = axes[1].imshow(Ceff_high_s, cmap='hot', vmin=0, vmax=0.2, aspect='auto')
draw_network_borders(axes[1], boundaries)
plt.colorbar(im1, ax=axes[1], label='GEC weight')
axes[1].set_title('HIGH Anhedonia — Group GEC', fontsize=12)
axes[1].set_xticks(mid_positions)
axes[1].set_xticklabels(display_order, rotation=45, ha='right', fontsize=8)
axes[1].set_yticks([])

# --- DIFFERENCE ---
diff_max = np.abs(Ceff_diff_s).max()
im2 = axes[2].imshow(Ceff_diff_s, cmap='RdBu_r', vmin=-diff_max, vmax=diff_max, aspect='auto')
draw_network_borders(axes[2], boundaries, color='gray')
plt.colorbar(im2, ax=axes[2], label='HIGH − LOW')
axes[2].set_title('Difference: HIGH − LOW', fontsize=12)
axes[2].set_xticks(mid_positions)
axes[2].set_xticklabels(display_order, rotation=45, ha='right', fontsize=8)
axes[2].set_yticks([])

fig.suptitle('Group-Level Effective Connectivity Matrices (Sorted by Yeo-7 Network)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# %% F3 — Edge scatter: LOW vs HIGH group GEC, coloured by SC strength
# SC .mat is HDF5 v7.3 — load from the companion CSV instead

SC = pd.read_csv(SC_CSV, header=None).values.astype(np.float64)  # (232, 232)
SC_norm = SC / SC.max() * 0.2  # normalise to same 0–0.2 scale as Ceff

# Upper triangle (no diagonal) — 26,796 edges
triu_i, triu_j = np.triu_indices(232, k=1)
low_vec  = Ceffgroup_LOW[triu_i, triu_j]
high_vec = Ceffgroup_HIGH[triu_i, triu_j]
sc_vec   = SC_norm[triu_i, triu_j]

sc_color = sc_vec / (sc_vec.max() + 1e-10)  # normalize 0–1 for colormap

r_gh = np.corrcoef(low_vec, high_vec)[0, 1]

fig, ax = plt.subplots(figsize=(7, 6))
sc_im = ax.scatter(low_vec, high_vec, c=sc_color, cmap='YlOrRd',
                   s=1, alpha=0.3, rasterized=True)
ax.plot([0, 0.2], [0, 0.2], 'k--', lw=0.8, label='y = x')
plt.colorbar(sc_im, ax=ax, label='SC strength (normalised)')
ax.set_xlabel('LOW Anhedonia Group GEC', fontsize=11)
ax.set_ylabel('HIGH Anhedonia Group GEC', fontsize=11)
ax.set_title(f'Edge-wise Comparison of Group GEC\n'
             f'Pearson r = {r_gh:.3f}  (n = {len(low_vec):,} edges)',
             fontsize=12)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
# %% F4 — Group-level trophic levels from Ceffgroup, bar chart per network
# These are single-value estimates (no per-subject variance), so no stat test is applicable.

tc_gl_low,  hl_gl_low  = compute_hierarchy(Ceffgroup_LOW)
tc_gl_high, hl_gl_high = compute_hierarchy(Ceffgroup_HIGH)

print(f'Group GEC trophic coherence:  LOW={tc_gl_low:.4f}  HIGH={tc_gl_high:.4f}')

yeo_colors = {
    'Vis': '#781286', 'SomMot': '#4682B4', 'DorsAttn': '#00760E',
    'SalVentAttn': '#C43AFA', 'Limbic': '#B8CF5E', 'Cont': '#E69422',
    'Default': '#CD3E4E', 'Subcortex': '#808080',
}

net_hl_low  = {net: np.mean(hl_gl_low[network_indices[net]])  for net in display_order}
net_hl_high = {net: np.mean(hl_gl_high[network_indices[net]]) for net in display_order}

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(display_order))
width = 0.35

for i, net in enumerate(display_order):
    col = yeo_colors[net]
    ax.bar(x[i] - width/2, net_hl_low[net],  width, color=col, edgecolor='black',
           linewidth=0.8, label='Low Anhedonia'  if i == 0 else '')
    ax.bar(x[i] + width/2, net_hl_high[net], width, color=col, edgecolor='black',
           linewidth=0.8, hatch='///', label='High Anhedonia' if i == 0 else '')

ax.set_xticks(x)
ax.set_xticklabels(display_order, rotation=45, ha='right', fontsize=11)
ax.set_ylabel('Mean Trophic Level (group-level)', fontsize=12)
ax.set_title('Group-Level Trophic Levels per Network\n'
             '(derived from Ceffgroup; no significance testing — single estimate per group)',
             fontsize=12)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
plt.show()

---
## Analysis G: SC→GEC Divergence per Subject

Each subject's individual GEC was optimised starting from the group-level GEC (which itself started from the structural connectivity matrix, SC). Here we quantify how much each subject's final GEC diverges from the group-average SC — a measure of how idiosyncratic the individual's effective connectivity is relative to the anatomical scaffold.

**Divergence metric:** `1 − Pearson r(Ceff_sub, SC_norm)` computed over all 26,796 upper-triangle edges. Higher = more divergence from SC.

**Hypothesis:** High anhedonia subjects may show greater or lesser divergence from SC, reflecting either more personalised (idiosyncratic) or more rigid (SC-constrained) effective connectivity patterns.

In [ ]:
# %% G1 — Compute SC–GEC divergence per subject
# SC_norm is already loaded from Analysis F (SC / SC.max() * 0.2)
# If running this section standalone, uncomment:
# SC = pd.read_csv(SC_CSV, header=None).values.astype(np.float64)
# SC_norm = SC / SC.max() * 0.2

# Use upper triangle to avoid diagonal zeros inflating the correlation
triu_mask = np.zeros((232, 232), dtype=bool)
triu_mask[np.triu_indices(232, k=1)] = True
sc_flat = SC_norm[triu_mask]  # (26796,)

div_LOW  = np.array([1 - pearsonr(Ceff_LOW[s][triu_mask],  sc_flat)[0] for s in range(NSUB_LOW)])
div_HIGH = np.array([1 - pearsonr(Ceff_HIGH[s][triu_mask], sc_flat)[0] for s in range(NSUB_HIGH)])

_, p_div_rs = ranksums(div_LOW, div_HIGH)
print('SC–GEC divergence (1 − Pearson r over upper triangle):')
print(f'  LOW  mean={div_LOW.mean():.4f}  std={div_LOW.std():.4f}')
print(f'  HIGH mean={div_HIGH.mean():.4f}  std={div_HIGH.std():.4f}')
print(f'  Ranksum p = {p_div_rs:.4f}')

In [ ]:
# %% G2 — OLS: divergence ~ anhedonia group + covariates

df_ols_div = pd.concat([
    pd.DataFrame({'subjectkey': ids_low,  'sc_gec_div': div_LOW,  'anhedonia_group': 'LOW'}),
    pd.DataFrame({'subjectkey': ids_high, 'sc_gec_div': div_HIGH, 'anhedonia_group': 'HIGH'}),
], ignore_index=True)
df_ols_div = df_ols_div.merge(
    demos[['subjectkey', 'Age_clean', 'sex', 'is_patient']], on='subjectkey', how='left')
df_ols_div['sex'] = df_ols_div['sex'].replace('O', np.nan)
df_ols_div['anhedonia_group'] = pd.Categorical(df_ols_div['anhedonia_group'], categories=['LOW', 'HIGH'])
df_ols_div['sex']             = pd.Categorical(df_ols_div['sex'],             categories=['F', 'M'])

# Full sample
res_div = smf.ols('sc_gec_div ~ anhedonia_group + Age_clean + sex + is_patient',
                   data=df_ols_div).fit()
coef_div = res_div.params['anhedonia_group[T.HIGH]']
p_div    = res_div.pvalues['anhedonia_group[T.HIGH]']
ci_div   = res_div.conf_int().loc['anhedonia_group[T.HIGH]']

print('OLS — SC–GEC Divergence ~ Anhedonia Group + covariates (full sample)')
print(f'  β = {coef_div:+.4f}  [{ci_div[0]:+.4f}, {ci_div[1]:+.4f}]')
print(f'  p = {p_div:.4f}  N = {int(res_div.nobs)}')
print()

# Patients only
df_pat_div = df_ols_div[df_ols_div['is_patient'] == 1].copy()
res_div_pat = smf.ols('sc_gec_div ~ anhedonia_group + Age_clean + sex', data=df_pat_div).fit()
coef_div_pat = res_div_pat.params['anhedonia_group[T.HIGH]']
p_div_pat    = res_div_pat.pvalues['anhedonia_group[T.HIGH]']

print(f'OLS — Patients only (N={int(res_div_pat.nobs)})')
print(f'  β = {coef_div_pat:+.4f}, p = {p_div_pat:.4f}')

In [ ]:
# %% G3 — Boxplot of SC–GEC divergence by group

fig, ax = plt.subplots(figsize=(5, 6))

bp = ax.boxplot([div_LOW, div_HIGH],
                tick_labels=['Low Anhedonia', 'High Anhedonia'],
                patch_artist=True)
bp['boxes'][0].set_facecolor(color_low)
bp['boxes'][1].set_facecolor(color_high)
for m in bp['medians']:
    m.set_color('black')
    m.set_linewidth(1.5)

ylo, yhi = ax.get_ylim()
ax.text(1.5, yhi - (yhi - ylo) * 0.05,
        f'OLS p = {p_div:.4f}', ha='center', va='top', fontsize=11)

ax.set_ylabel('SC–GEC Divergence  (1 − Pearson r)', fontsize=11)
ax.set_title('Individual GEC Divergence from\nStructural Connectivity', fontsize=12)
fig.tight_layout()
plt.show()

In [ ]:
# %% G2_DX — Sensitivity: SC–GEC Divergence ~ Anhedonia Group + Dx_broad (HC ref)

df_ols_div_dx = df_ols_div.merge(
    demos_dx[['subjectkey', 'Dx_broad']], on='subjectkey', how='left')
df_ols_div_dx['Dx_broad'] = pd.Categorical(
    df_ols_div_dx['Dx_broad'],
    categories=['HC', 'Mood', 'Anxiety_Trauma', 'Other']
)

res_div_dx = smf.ols(
    'sc_gec_div ~ anhedonia_group + Age_clean + sex + Dx_broad',
    data=df_ols_div_dx.dropna(subset=['Dx_broad'])
).fit()
coef_div_dx = res_div_dx.params['anhedonia_group[T.HIGH]']
p_div_dx    = res_div_dx.pvalues['anhedonia_group[T.HIGH]']
ci_div_dx   = res_div_dx.conf_int().loc['anhedonia_group[T.HIGH]']

print('OLS — SC–GEC Divergence ~ Anhedonia Group + Dx_broad (HC ref)')
print(f'  β = {coef_div_dx:+.4f}  [{ci_div_dx[0]:+.4f}, {ci_div_dx[1]:+.4f}]')
print(f'  p = {p_div_dx:.4f}  N = {int(res_div_dx.nobs)}')


**interpretation**  
β = -0.0080 — High anhedonia group has slightly lower SC–GEC divergence than Low, after controlling for Age, Sex, and diagnostic category  
p = 0.039 — significant at the uncorrected α = 0.05 threshold  
CI = [-0.0155, -0.0004] — just excludes zero, so it's a marginal but real effect  
N = 206 — 4 subjects dropped (likely the sex='O' cases or missing Dx)  